In [5]:
import os

print("Direktori kerja:", os.getcwd())
print("Path data:", os.path.exists("data/raw/kredit-sumatera.csv"))
print("Path alternatif:", os.path.exists("../data/raw/kredit-sumatera.csv"))

Direktori kerja: c:\Users\Asus\My Drive\HR\Prak MLOPS\MLOps\notebooks
Path data: False
Path alternatif: False


In [6]:
from pathlib import Path

root = Path.cwd().parent

print("Folder utama:", root)
print("\nIsi folder data:")

data_folder = root / "data"

if data_folder.exists():
    for item in data_folder.iterdir():
        print(item)
else:
    print("Folder data tidak ditemukan.")

print("\nPencarian file CSV:")
for file in root.rglob("kredit*.csv"):
    if ".venv" not in str(file):
        print(file)

Folder utama: c:\Users\Asus\My Drive\HR\Prak MLOPS\MLOps

Isi folder data:
c:\Users\Asus\My Drive\HR\Prak MLOPS\MLOps\data\raw

Pencarian file CSV:
c:\Users\Asus\My Drive\HR\Prak MLOPS\MLOps\data\raw\kredit_sumatera.csv


In [7]:
import pandas as pd

df = pd.read_csv("../data/raw/kredit_sumatera.csv")

print(df.shape)
print(df.head())

# Pertanyaan 1: seberapa tidak seimbang targetnya?
print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")

# Pertanyaan 2: apakah populasi berubah sepanjang waktu?
print(
    pd.crosstab(
        df.periode.str[:4],
        df.segmen,
        normalize="index"
    ).round(3)
)

# Pertanyaan 3: kolom mana yang punya nilai hilang, dan mengapa?
hilang = df.isna().mean().sort_values(ascending=False)
print(hilang[hilang > 0].round(4))

print(
    df.groupby("segmen")["rata_saldo_90h"]
    .apply(lambda s: s.isna().mean())
    .round(3)
)

# Pertanyaan 4: adakah kolom yang mencurigakan?
print(
    df.groupby("gagal_bayar")["status_restrukturisasi"]
    .apply(lambda s: s.notna().mean())
)

(32400, 19)
    pengajuan_id tanggal_pengajuan  periode      segmen        wilayah  \
0  2022-01-00302        2022-01-01  2022-01  wiraswasta      Pesawaran   
1  2022-01-00769        2022-01-01  2022-01  wiraswasta      Pringsewu   
2  2022-01-00235        2022-01-01  2022-01      petani      Pringsewu   
3  2022-01-00661        2022-01-01  2022-01      petani      Pesawaran   
4  2022-01-00325        2022-01-01  2022-01  wiraswasta  Lampung Barat   

    kanal  usia  tanggungan  lama_usaha_tahun  penghasilan_bulanan  \
0    agen    28           3               6.2            6760000.0   
1    agen    48           1               2.4            3950000.0   
2    agen    44           4               2.2            4000000.0   
3  cabang    45           1               2.9            7740000.0   
4  cabang    46           2              28.2           11000000.0   

   ragam_penghasilan  plafon_diajukan  tenor_bulan  bunga_tahunan  \
0              0.282       14560000.0           12   

In [2]:
import sys
print(sys.executable)

c:\Users\Asus\My Drive\HR\Prak MLOPS\MLOps\mlops-kredit-sumatera\.venv\Scripts\python.exe


In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Membuat fitur rasio angsuran
df["rasio_angsuran"] = (
    df.angsuran_bulanan / df.penghasilan_bulanan
)

# Menentukan variabel prediktor
num = [
    "usia", "tanggungan", "lama_usaha_tahun",
    "penghasilan_bulanan", "ragam_penghasilan",
    "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
    "rasio_angsuran", "rata_saldo_90h",
    "jumlah_tunggakan_12b"
]

# Menggabungkan fitur numerik dan kategorik
X = df[num].join(
    pd.get_dummies(df[["segmen", "wilayah", "kanal"]])
)

# Menentukan target
y = df.gagal_bayar

# Pembagian MENURUT WAKTU, bukan acak
latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

# Membuat model
m = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.06,
    max_iter=250,
    random_state=42
)

# Melatih model
m.fit(X[latih], y[latih])

# Prediksi probabilitas gagal bayar
skor = m.predict_proba(X[uji])[:, 1]

# Evaluasi model menggunakan ROC-AUC
print("AUC out-of-time:", round(
    roc_auc_score(y[uji], skor), 4
))

AUC out-of-time: 0.774


In [9]:
from sklearn.model_selection import train_test_split
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
max_iter=250, random_state=42).fit(Xa, ya)
print("AUC pembagian acak:", round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC pembagian acak: 0.7629


In [10]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

print("Scikit-learn:", sklearn.__version__)

Xa, Xb, ya, yb = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

m2 = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.06,
    max_iter=250,
    random_state=42
)

m2.fit(Xa, ya)

print(
    "AUC pembagian acak:",
    round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4)
)

Scikit-learn: 1.9.1
AUC pembagian acak: 0.7629
